# TBBT - adnotacje kandydujące

Przenosi opisy `v` z TVR na oś odcinka, dopasowuje je do masek i zapisuje pliki do wczytania w adnotatorze. Odcinki bez napisów, bez nagrania albo bez masek pomija i wypisuje, więc można go uruchamiać w miarę postępu.

**Wymaga:** `tbbt_01_episode_selection.ipynb` oraz przygotowanych masek (procedura niżej). Napisy odcinków leżą w `data/subs/tbbt` (`tbbt_s01e15.srt`, jak plik wideo), klipowe w `data/interim/tbbt/tvqa_subtitles`.

**Zapisuje:**

| plik | co zawiera | gdzie powstaje |
|---|---|---|
| `data/interim/tbbt/work/tbbt_video.csv` | rozdzielczość, fps i długość każdego pliku `.mp4` | blok 1 |
| `data/interim/tbbt/work/tbbt_clip_mapping.csv` | przesunięcie każdego klipu TVR względem odcinka | blok 2 |
| `data/interim/tbbt/tbbt_annotations.csv` | rejestr zbiorczy: wszystkie opisy `v` z czasami oryginalnymi i werdyktem | blok 3 |
| `data/annotations/tbbt/candidates/tbbt_<odcinek>_intervals_off.csv`<br>`..._forward.csv` | pliki do wczytania w adnotatorze, po dwa warianty na odcinek | blok 4 |
| `data/interim/tbbt/work/tbbt_fitting_<wariant>.csv` | przesunięcie i flagi każdej adnotacji przy dopasowaniu do masek | blok 4 |

Pliki `_intervals.csv` powstają w `candidates/` i są tam bez ceregieli nadpisywane - to wynik obliczenia, nie praca ręczna. Zweryfikowane pliki przenosisz ręcznie do `data/annotations/tbbt/`; ten notatnik do tego katalogu nigdy nie pisze, czyta z niego tylko po to, żeby zgłosić rozjazd.

**Dalej:** weryfikacja plików z `candidates/` w `tools/interval-annotator`, przeniesienie gotowych do `data/annotations/tbbt/`, a potem `tbbt_03_annotations.ipynb`.

---

## Jak przygotować maski

Maski to czołówka i napisy końcowe, czyli fragmenty wycinane z korpusu. Nie mają adnotacji, a w każdym odcinku wyglądają tak samo, więc zostawione w korpusie dałyby po dwadzieścia bliźniaczych fragmentów, które myliłyby się nawzajem i zaniżały wynik każdego wariantu potoku po równo, zamazując różnice między badanymi komponentami.

Notatnik ich nie wykrywa i nie sprawdza, tylko czyta gotowe pliki, po jednym na odcinek:

1. Otworzyć `tools/interval-annotator/index.html` w przeglądarce.
2. **Choose file** -> `data/processed/tbbt/tbbt_<odcinek>.mp4`.
3. Odznaczyć *Lock masks* i przyciskiem **Add mask** dodać czołówkę oraz napisy końcowe. Jeśli po napisach jest jeszcze scenka, napisy dostają własną maskę, a scenka zostaje w korpusie.
4. **Export CSV** do `data/annotations/tbbt/masks/`, pod nazwą `tbbt_<odcinek>_intervals.csv` (adnotator proponuje ją sam).

Plik może zawierać same maski; adnotacje dokłada blok 4. Osobny katalog `masks/` jest po to, żeby blok 4 nie mógł tej pracy nadpisać: zapisuje on wyłącznie do `candidates/`.

In [ ]:
SERIES        = "tbbt"
EPISODE_COUNT = 24       # reads tbbt_selection_<COUNT>.csv
CORRECTIONS   = {}       # {"s09e12": -2.0} - hand-checked offset per episode

import csv
import importlib
import re
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import adjust
from src.annotation import intervals as iv
from src.annotation import ranges as rg
from src.annotation import registry as reg
from src.annotation import tbbt_time_mapping as tm
from src.utils import settings
from src.utils import video_probe as vp

for module in (iv, rg, reg, adjust, tm, settings):
    importlib.reload(module)

DATA_DIR   = ROOT / "data" / "interim" / SERIES
WORK_DIR   = DATA_DIR / "work"
VIDEO_DIR  = ROOT / "data" / "processed" / SERIES
SUBS_DIR   = ROOT / "data" / "subs" / SERIES
CLIPS_DIR  = DATA_DIR / "tvqa_subtitles"

MASKS_DIR        = ROOT / "data" / "annotations" / SERIES / "masks"
CANDIDATES_DIR   = ROOT / "data" / "annotations" / SERIES / "candidates"
VERIFIED_DIR     = ROOT / "data" / "annotations" / SERIES
DESCRIPTIONS_CSV = DATA_DIR / f"{SERIES}_descriptions_v.csv"
REGISTER_CSV     = DATA_DIR / f"{SERIES}_annotations.csv"
MAPPING_CSV      = WORK_DIR / f"{SERIES}_clip_mapping.csv"

MIN_DURATION, MAX_DURATION = settings.MIN_DURATION, settings.MAX_DURATION

with open(DATA_DIR / f"{SERIES}_selection_{EPISODE_COUNT}.csv",
          encoding="utf-8-sig", newline="") as f:
    SPLIT = {r["episode"]: r["split"] for r in csv.DictReader(f, delimiter=";")}
VIDEO_FILE = {ep: f"{SERIES}_{ep}.mp4" for ep in SPLIT}

_EPISODE = re.compile(r"s\d{2}e\d{2}", re.IGNORECASE)
_video_csv = WORK_DIR / f"{SERIES}_video.csv"
DURATION = {}
if _video_csv.exists():
    with open(_video_csv, encoding="utf-8-sig", newline="") as f:
        DURATION = {m.group(0).lower(): float(r["duration"])
                    for r in csv.DictReader(f, delimiter=";")
                    if (m := _EPISODE.search(r["file"]))}

EPISODES = [ep for ep in sorted(SPLIT)
            if tm.subtitle_path(SUBS_DIR, ep, SERIES).exists()
            and (VIDEO_DIR / VIDEO_FILE[ep]).exists()]

# the clip offsets are expensive to compute, so they are read back from the file
# cell 2 writes; `results` has the shape that cell 3 expects
results = []
if MAPPING_CSV.exists():
    import collections as _collections

    _clips = _collections.defaultdict(list)
    with open(MAPPING_CSV, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f, delimiter=";"):
            for key in ("offset", "spread", "jump_start", "jump_end",
                        "offset_before", "offset_after"):
                if key in r:
                    r[key] = float(r[key]) if r[key] not in ("", None) else None
            r["jump"] = (r.get("jump") == "yes")
            _clips[r["episode"]].append(r)
    results = [{"episode": ep, "clips": v} for ep, v in _clips.items()]

rows = reg.load(REGISTER_CSV) if REGISTER_CSV.exists() else []

mask_items = {}
for ep in EPISODES:
    path = MASKS_DIR / iv.file_name(SERIES, ep)
    items = [i for i in iv.read(path) if i.is_mask] if path.exists() else []
    if items:
        mask_items[ep] = items

print(f"{SERIES}: {len(SPLIT)} episodes selected, {len(EPISODES)} with subtitles"
      f" and video, {len(mask_items)} with masks")
print(f"clip offsets for {len(results)} episodes, register {len(rows)} rows,"
      f" durations for {len(DURATION)} episodes")

## 1. Pomiar plików wideo

**Zapisuje:** `data/interim/tbbt/work/tbbt_video.csv` - rozdzielczość, fps, liczba klatek i długość każdego pliku odcinka. Długość jest potrzebna do maski napisów końcowych i do zakresów korpusu.

In [ ]:
SERIES    = "tbbt"
DATA_DIR  = ROOT / "data" / "interim" / SERIES
WORK_DIR  = DATA_DIR / "work"
VIDEO_DIR = ROOT / "data" / "processed" / SERIES

rows = vp.probe_videos(VIDEO_DIR)
print(vp.format_summary(rows))                 # table + uniformity verdict
vp.save_csv(rows, WORK_DIR / f"{SERIES}_video.csv")

info = vp.total_duration(VIDEO_DIR)
print(f"\ntotal duration: {info['total']} s = {info['hhmmss']} (hh:mm:ss)")
print(f"saved -> {(WORK_DIR / f'{SERIES}_video.csv').relative_to(ROOT)}")

## 2. Przesunięcia klipów

**Zapisuje:** `data/interim/tbbt/work/tbbt_clip_mapping.csv` - jeden wiersz na klip TVR z liczbą sekund, którą trzeba dodać do czasu adnotacji, żeby trafić w oś pliku `.mp4`. Plik jest nadpisywany.

Przesunięcie liczone jest **osobno dla każdego klipu**, a nie raz na odcinek. Klipy nie pokrywają odcinka w całości ani nie stykają się ze sobą: w `s01e01` pierwszy segment ma przesunięcia 0, 61, 124, 182, 241, 301 s, a kolejny zaczyna się dopiero od 382 s - między nimi zostaje materiał, którego autorzy TVQA nie wycięli.

Punktem odniesienia są napisy: ten sam dialog jest w napisach klipowych TVQA i w napisach do odcinka, tylko w innym momencie. Procedura (`src/annotation/tbbt_time_mapping.py`):

1. **Czyszczenie tekstu.** Z obu stron usuwane jest to, czym oba zapisy się różnią: znaczniki HTML, didaskalia w nawiasach, imiona mówców, interpunkcja i wielkość liter. Zostaje sama lista słów.
2. **Zamiana na ciąg słów.** Porównywane są pojedyncze słowa, a nie linijki, bo obie strony inaczej łamią kwestie na wiersze. Każde słowo dostaje własny czas: napis wyświetlany od 12,0 s do 14,0 s i złożony z czterech słów daje słowa w 12,25 s, 12,75 s, 13,25 s i 13,75 s. Bez tego wszystkie słowa jednej linijki miałyby ten sam czas i dopasowanie w środku długiej kwestii byłoby warte tyle, co dopasowanie na jej początku.
3. **Dopasowanie.** `difflib` znajduje najdłuższe wspólne fragmenty obu ciągów, z zachowaniem kolejności. Fragmenty krótsze niż 4 słowa są odrzucane, bo krótkie zwroty powtarzają się w odcinku wielokrotnie.
4. **Kotwice.** Każde dopasowane słowo daje parę czasów: kiedy padło w klipie i kiedy w odcinku. Gdy słowo jest **pierwszym słowem linijki po obu stronach**, brane są surowe czasy z plików SRT - nie trzeba wtedy zgadywać, w którym momencie linijki to słowo padło, więc taka kotwica jest dokładna i to na niej liczone jest przesunięcie. Pozostałe kotwice korzystają z czasów wyliczonych w kroku 2 i służą jako zapas, gdy kotwic dokładnych jest za mało.
5. **Przesunięcie.** Dla każdej kotwicy liczona jest różnica obu czasów. Brana jest najliczniejsza grupa różnic mieszczących się w oknie ±1 s i z niej mediana - pojedyncza pomyłka (zdanie, które pada w odcinku dwa razy) wypada wtedy poza grupę i nie psuje wyniku.

Sprawdzane jest też, czy przesunięcie nie zmienia się **w środku** klipu: nagranie w tym projekcie bywa krótsze od źródła TVQA o kilka sekund na przejściu między scenami, więc klip przechodzący przez taką granicę wymaga dwóch różnych przesunięć. Takie klipy liczy kolumna `jumps`, a ich adnotacje dostają flagę `uncertain_mapping`.

**Jak czytać kolumnę `accuracy`.** Mówi ona, z jaką dokładnością znane jest przesunięcie:

- **`frame-exact`** - przesunięcia wszystkich klipów odcinka wypadają na siatce co 1 s (TVQA cięło klipy na pełnych sekundach). Tak jest, gdy oba zestawy napisów mają wspólne pochodzenie czasowe; przesunięcie jest wtedy pewne co do milisekund.
- **`~0,3 s`** - napisy powstały niezależnie, więc ta sama kwestia ma w obu plikach nieco inny znacznik czasu. Wypisana liczba to mediana różnic: dla każdej linijki klipu o co najmniej 5 słowach liczony jest czas na osi odcinka, ta linijka jest szukana w napisach odcinka w oknie ±4 s, i zapisywana jest różnica między czasem znalezionym a policzonym. **To nie jest błąd algorytmu, tylko granica dokładności, jaką dwa niezależnie zrobione zestawy napisów w ogóle pozwalają osiągnąć.**
- **`~nan s`** - żadna linijka nie została potwierdzona: albo klipy odcinka w ogóle nie dostały przesunięcia (za mało kotwic), albo sprawdzenie nic nie znalazło w oknie. To sygnał do obejrzenia odcinka, a nie wynik dobry.

**Poprawki ręczne (`CORRECTIONS`).** Jest jeden błąd, którego z napisów nie da się wykryć: gdy same napisy do odcinka są przesunięte względem obrazu. Wtedy oba zestawy zgadzają się ze sobą, przesunięcie wychodzi "frame-exact", a adnotacje i tak wypadają obok. Widać to dopiero przy oglądaniu, dlatego poprawka jest wpisywana ręcznie na górze komórki. Obecnie jest tam jedna pozycja: `s09e12` o -2 s.

In [ ]:
# only episodes that have both subtitles (mapping) and a video file (mask)
EPISODES = [e for e in sorted(SPLIT)
            if tm.subtitle_path(SUBS_DIR, e, SERIES).exists()
            and (VIDEO_DIR / VIDEO_FILE[e]).exists()]
NO_SRT = [e for e in sorted(SPLIT) if not tm.subtitle_path(SUBS_DIR, e, SERIES).exists()]
NO_MP4 = [e for e in sorted(SPLIT) if not (VIDEO_DIR / VIDEO_FILE[e]).exists()]
if NO_SRT:
    print(f"TO RIP - no subtitles ({len(NO_SRT)}): {', '.join(NO_SRT)}")
    print(f"expected name: {SERIES}_<episode>.srt, the same as the video file")
if NO_MP4:
    print(f"TO RIP - no video ({len(NO_MP4)}): {', '.join(NO_MP4)}")
if NO_SRT or NO_MP4:
    print(f"computing for now {len(EPISODES)} of {len(SPLIT)} selected episodes\n")

results = tm.map_episodes(SUBS_DIR, CLIPS_DIR, EPISODES, series=SERIES)
corrected = tm.apply_corrections(results, CORRECTIONS)

header = f"{'episode':<9}{'split':<7}{'clips':>7}{'accuracy':>14}{'jumps':>8}{'correction':>12}"
print(header)
print("-" * len(header))
for r in results:
    ep = r["episode"]
    checks = tm.check_by_text(tm.subtitle_path(SUBS_DIR, ep, SERIES),
                              CLIPS_DIR, r["clips"])
    acc = "frame-exact" if r["grid"]["R"] >= 0.9 else f"~{checks['error_med']:.1f} s"
    jumps = sum(1 for c in r["clips"] if c.get("jump"))
    corr = CORRECTIONS.get(ep, 0.0)
    print(f"{ep:<9}{SPLIT[ep]:<7}{len(r['clips']):>7}{acc:>14}{jumps:>8}"
          f"{(f'{corr:+.1f} s' if corr else ''):>12}")

flat = tm.flat_rows(results)
for r in flat:
    r["split"] = SPLIT[r["episode"]]
COLUMNS = ["vid_name", "episode", "split", "segment", "clip_no", "offset",
           "correction", "anchors", "spread", "jump", "jump_start", "jump_end",
           "offset_before", "offset_after"]
tm.save_csv(flat, WORK_DIR / f"{SERIES}_clip_mapping.csv", COLUMNS)

checks = tm.check(results)
print(f"\nmapped {checks['mapped']} of {checks['clips']} clips")
for s in ("dev", "test"):
    o = [r for r in results if SPLIT[r["episode"]] == s]
    print(f"  {s:<5}{len(o):>3} episodes, {sum(len(r['clips']) for r in o):>4} clips")
if corrected:
    print(f"manual offset correction: {CORRECTIONS} -> {corrected} clips")
print("accuracy 'frame-exact' = our subtitles and the TVQA subtitles have identical")
print("timestamps, so the offsets land on whole seconds (error ~1 ms);")
print("in the remaining episodes the subtitles were made independently and the error is given.")
print(f"saved -> {(WORK_DIR / f'{SERIES}_clip_mapping.csv').relative_to(ROOT)}")

## 3. Rejestr zbiorczy adnotacji

**Zapisuje:** `data/interim/tbbt/tbbt_annotations.csv` - każdy opis `v` z czasem na osi odcinka i werdyktem.

**Czasy są oryginalne.** Rejestr zapisuje to, co wyszło z kotwiczenia klipów, bez dopasowywania do masek - dopasowanie należy do kroku 4 i różni się między wariantami, więc nie miałoby tu jednej wartości. Kolumny `source_start` i `source_end` zapamiętują te czasy na stałe; po naniesieniu weryfikacji w `tbbt_03` różnica względem `start` i `end` pokazuje wszystko, co stało się z adnotacją po drodze.

**Rejestr powstaje od zera przy każdym uruchomieniu** - to wynik obliczenia z bieżącymi progami. Decyzje z adnotatora nanosi dopiero `tbbt_03_annotations.ipynb`, czytając zweryfikowane pliki `_intervals.csv`; nic tu nie ginie, bo praca ręczna mieszka w tamtych plikach.

Werdykt jest tylko dwojakiego rodzaju, liczony z długości:

| powód | kiedy |
|---|---|
| `too_short` | krócej niż `MIN_DURATION` (1,25 s, krok próbkowania klatek) - nie trafi w żadną próbkowaną klatkę |
| `too_long` | dłużej niż `MAX_DURATION` (15 s) - rozłoży się na kilka fragmentów, więc "poprawna odpowiedź" przestaje być jednym fragmentem |

Masek ten krok nie zna, więc nie ma tu werdyktu `in_mask`. Adnotacje leżące w masce wyłapuje adnotator, a `tbbt_03` wypisuje je jako ostrzeżenie.

Kolumna `flags` niesie tylko `uncertain_mapping` - adnotacja przechodzi przez miejsce, w którym nagranie różni się od źródła TVQA. Flagi dopasowania do masek (`shifted`, `trimmed`, `shift_failed`, `spans_mask`) opisują krok 4 i lądują w plikach `work/tbbt_fitting_<wariant>.csv`.

Progi `MIN_DURATION` i `MAX_DURATION` pochodzą z `src/utils/settings.py` - jednego miejsca wspólnego dla obu seriali, skryptów i notatników.

In [ ]:
offsets = {c["vid_name"]: c for r in results for c in r["clips"]}      # cell 2
with open(DESCRIPTIONS_CSV, encoding="utf-8-sig", newline="") as f:
    descriptions = [r for r in csv.DictReader(f, delimiter=";")
                    if r["episode"] in EPISODES]

# Every description on the episode axis, with the ORIGINAL times: fitting them to
# the masks belongs to the next cell and differs per variant, so the register
# would have no single value to write.
rows, unanchored = [], []
for r in descriptions:
    episode, desc_id = r["episode"], int(r["desc_id"])
    start, end, confidence = tm.map_ts((float(r["ts_start"]), float(r["ts_end"])),
                                       offsets[r["vid_name"]])
    if start is None:
        # the clip could not be anchored on the episode axis, so the annotation
        # has no time at all and cannot enter the register
        unanchored.append((episode, r["vid_name"]))
        continue
    length = end - start
    status, reason = reg.STATUS_ACCEPTED, ""
    if length < MIN_DURATION:
        status, reason = reg.STATUS_REJECTED, reg.REASON_TOO_SHORT
    elif length > MAX_DURATION:
        status, reason = reg.STATUS_REJECTED, reg.REASON_TOO_LONG

    rows.append(reg.row(
        iv.external_id(SERIES, episode, desc_id), episode, SPLIT[episode],
        VIDEO_FILE[episode], start, end, r["desc"],
        desc_id=desc_id, vid_name=r["vid_name"],
        ts_start=float(r["ts_start"]), ts_end=float(r["ts_end"]),
        status=status, reason=reason,
        flags=[reg.FLAG_UNCERTAIN] if confidence == "uncertain" else []))

reg.save(rows, REGISTER_CSV)

header = (f"{'episode':<9}{'split':<7}{'total':>7}{'accepted':>10}"
          + "".join(f"{r:>11}" for r in reg.REASONS))
print(header)
print("-" * len(header))
for s in reg.stats(rows, EPISODES):
    print(f"{s['episode']:<9}{s['split']:<7}{s['total']:>7}{s['accepted']:>10}"
          + "".join(f"{s[r]:>11}" for r in reg.REASONS))
print("-" * len(header))
for t in reg.totals(rows):
    print(f"{t['split']:<9}{'':<7}{t['total']:>7}{t['accepted']:>10}"
          + "".join(f"{t[r]:>11}" for r in reg.REASONS))

uncertain = sum(1 for r in rows if reg.FLAG_UNCERTAIN in r["flags"])
print(f"\nuncertain_mapping: {uncertain}")
if unanchored:
    clips = sorted({v for _, v in unanchored})
    print(f"NOT IN THE REGISTER: {len(unanchored)} descriptions from {len(clips)}"
          " clips the subtitles did not anchor:")
    print("   " + ", ".join(clips[:8]) + (" ..." if len(clips) > 8 else ""))
print(f"saved -> {REGISTER_CSV.relative_to(ROOT)}")

## 4. Pliki dla adnotatora - dwa warianty na odcinek

**Zapisuje:** `data/annotations/tbbt/candidates/tbbt_<odcinek>_intervals_off.csv` oraz `..._forward.csv` - po dwa pliki na odcinek, w formacie `id;type;start;end;desc;tags`, który adnotator wczytuje wprost. Obok, w `work/tbbt_fitting_<wariant>.csv`, ląduje szczegół dopasowania: przesunięcie każdej adnotacji i jej flagi.

**Skąd dwa warianty.** Okna TVR często zahaczają o maskę i trzeba je do niej dopasować. Reguła zależy od strony, z której zahaczają (`src/annotation/adjust.py`):

| sytuacja | co się dzieje | flaga |
|---|---|---|
| adnotacja **zaczyna się** w masce i wychodzi poza nią | znacznik otworzył się za wcześnie - cały przedział jest **przesuwany** za maskę, z zachowaniem długości | `shifted` |
| adnotacja **kończy się** w masce | ogon to dopełnienie okna TVR, nie zdarzenie - przedział jest **docinany** do bliższej krawędzi | `trimmed` |
| przesunięcie nie ma gdzie wylądować | czasy zostają nietknięte, decyzja wraca do człowieka | `shift_failed` |
| adnotacja **przechodzi przez całą maskę** | widoczne zdarzenie nie może trwać przez zmianę sceny, więc znacznik jest źle umiejscowiony | `spans_mask` |
| adnotacja **leży w całości w masce** | nie ma czego ratować - nie trafia do pliku | - |

Warianty różnią się tym, co dzieje się z **pozostałymi** adnotacjami odcinka:

| wariant | działanie |
|---|---|
| `off` | każda adnotacja dopasowywana osobno |
| `forward` | zmierzone przesunięcie `x` jest dodawane do **każdej późniejszej** adnotacji odcinka, zanim zostanie dopasowana |

`forward` ratuje odcinek, w którym znaczniki są systematycznie za wczesne - także te adnotacje, które w żadną maskę nie trafiły. Tam, gdzie nie są, przesuwa czasy, które były już dobre. **Który wariant pasuje, jest własnością odcinka**, więc oba powstają zawsze, a wybierasz przenosząc plik.

**Pliki powstają za każdym razem od nowa** i nie ma tu żadnego przełącznika nadpisywania. Do `data/annotations/tbbt/` ten notatnik nie pisze; komórka zagląda tam wyłącznie po to, żeby zgłosić, które odcinki są już zweryfikowane i czym różnią się od świeżych kandydatów.

**Nic nie jest tu odsiewane.** Adnotacje krótsze niż `MIN_DURATION` i dłuższe niż `MAX_DURATION` też trafiają do plików - adnotator zaznaczy je na czerwono. Jeśli poprawisz czasy tak, że mieszczą się w progach, `tbbt_03` zmierzy je ponownie i wpuści z powrotem.

Przed zapisem każdy plik przechodzi kontrolę: identyfikatory zgodne z konwencją i niepowtarzalne, dodatnia długość, **maski nienachodzące na siebie nawzajem**. Adnotacja nachodząca na maskę **nie jest** błędem, tylko ostrzeżeniem.

In [ ]:
import collections

CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

FITTING_COLUMNS = ["episode", "annotation_id", "variant", "source_start",
                   "source_end", "start", "end", "offset", "flags"]

# `mask_items` comes from the setup cell; an episode without masks is skipped,
# because there would be nothing to fit its annotations to
no_masks = [ep for ep in EPISODES if ep not in mask_items]
if no_masks:
    print(f"no masks ({len(no_masks)}): {', '.join(no_masks)}")
    print(f"mark them in the annotator and export to {MASKS_DIR.relative_to(ROOT)}\n")
WITH_MASKS = [ep for ep in EPISODES if ep in mask_items]

# Two classes of mask. The structural ones (logo, titles, credits) define the
# corpus ranges; the transitions do not cut a range, they become holes of the
# content axis. An annotation may sit on neither, so fitting works against both.
by_class = {ep: rg.split_masks(mask_items[ep]) for ep in WITH_MASKS}
corpus   = {ep: rg.from_masks(by_class[ep][0], DURATION[ep]) for ep in WITH_MASKS}
all_masks = {ep: rg.merge(by_class[ep][0] + by_class[ep][1]) for ep in WITH_MASKS}

by_episode = {}
for r in rows:
    if r["episode"] in WITH_MASKS:
        by_episode.setdefault(r["episode"], []).append(r)

summary, refused, warned, fitting = {}, [], {}, {}
for variant in adjust.PROPAGATION_MODES:
    fitting[variant] = []
    written = 0
    for ep in WITH_MASKS:
        entries = [(r["annotation_id"], float(r["start"]), float(r["end"]))
                   for r in by_episode.get(ep, [])]
        fitted = adjust.fit_episode(entries, all_masks[ep], corpus[ep], variant)

        items = list(mask_items[ep])      # masks exactly as marked in the annotator
        for r in by_episode.get(ep, []):
            result = fitted[r["annotation_id"]]
            if result is None:            # entirely inside a mask - nothing to save
                continue
            start, end, flags, offset = result
            items.append(iv.Interval(r["annotation_id"], iv.EVENT, start, end, r["desc"]))
            fitting[variant].append({
                "episode": ep, "annotation_id": r["annotation_id"], "variant": variant,
                "source_start": r["source_start"], "source_end": r["source_end"],
                "start": start, "end": end, "offset": offset, "flags": " ".join(flags)})
        items.sort(key=iv.sort_key)

        problems = iv.validate(items, SERIES, ep)
        if problems:
            refused.append((ep, variant, problems))
            continue
        overlaps = iv.warnings(items)
        if overlaps:
            warned.setdefault((ep, variant), overlaps)
        name = f"{iv.file_name(SERIES, ep)[:-4]}_{variant}.csv"
        iv.write(CANDIDATES_DIR / name, items)
        written += 1

    counts = collections.Counter(f for row in fitting[variant]
                                 for f in row["flags"].split())
    summary[variant] = {"files": written, "flags": dict(counts),
                        "dropped": sum(len(by_episode.get(ep, [])) for ep in WITH_MASKS)
                                   - len(fitting[variant])}
    with open(WORK_DIR / f"{SERIES}_fitting_{variant}.csv", "w",
              encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FITTING_COLUMNS, delimiter=";")
        writer.writeheader()
        writer.writerows(fitting[variant])

for variant, s in summary.items():
    print(f"{variant:<9}{s['files']:>3} files   dropped (inside a mask): "
          f"{s['dropped']:>3}   flags: {s['flags']}")

moved = sorted(ep for ep in WITH_MASKS
               if (VERIFIED_DIR / iv.file_name(SERIES, ep)).exists())
if moved:
    print(f"\nalready verified in {VERIFIED_DIR.relative_to(ROOT)}: {', '.join(moved)}")
    for ep in moved:
        on_disk = {i.id for i in iv.read(VERIFIED_DIR / iv.file_name(SERIES, ep))
                   if not i.is_mask}
        fresh = {r["annotation_id"] for r in by_episode.get(ep, [])}
        extra, absent = len(on_disk - fresh), len(fresh - on_disk)
        if extra or absent:
            print(f"   {ep}: {extra} in the verified file but not generated now, "
                  f"{absent} generated now but absent there")

for ep, variant, problems in refused:
    print(f"\nNOT WRITTEN {ep} ({variant}): {len(problems)} problems")
    for problem in problems[:5]:
        print(f"   {problem}")

if warned:
    total = sum(len(w) for w in warned.values())
    print(f"\nannotations sitting on a mask ({total}) - not an error, they load into"
          " the annotator marked and wait for a decision:")
    for (ep, variant), overlaps in sorted(warned.items()):
        print(f"   {ep} ({variant}): {len(overlaps)}  e.g. {overlaps[0]}")

print(f"\ncandidates -> {CANDIDATES_DIR.relative_to(ROOT)}")
print(f"fitting detail -> {WORK_DIR.relative_to(ROOT)}/{SERIES}_fitting_<variant>.csv")
print(f"move the chosen variant to {VERIFIED_DIR.relative_to(ROOT)}, dropping the suffix")

## Co dalej

1. Otworzyć `tools/interval-annotator/index.html` w przeglądarce.
2. Wskazać plik `data/processed/tbbt/tbbt_<odcinek>.mp4`, a potem **Import CSV** i jeden z dwóch kandydatów odcinka z `data/annotations/tbbt/candidates/`. Warto obejrzeć oba - różnią się czasami adnotacji, które w żadną maskę nie trafiły.
3. Przejrzeć adnotacje. Usuwać te, które nie opisują tego, co widać - na przykład mówiące wyłącznie o treści wypowiedzi ("Randall says that Penny sent him cigarettes"). Opisy, w których obok mówienia jest jakakolwiek czynność widoczna - gest, zmiana pozycji, przedmiot w ręku, reakcja twarzy - zostają, bo model ma się czego uchwycić. Adnotacje zaznaczone na czerwono jako *outside duration* można poprawić zamiast kasować.
4. Poprawić maski, jeśli po obejrzeniu granica wygląda inaczej: odznaczyć **Lock masks**. Poprawka wchodzi do pliku wynikowego; kopia w `masks/` zostaje bez zmian i przyda się przy kolejnym generowaniu kandydatów.
5. **Export CSV** i zapisać **do `data/annotations/tbbt/`**, pod nazwą bez przyrostka: `tbbt_<odcinek>_intervals.csv`. To ten katalog czyta `tbbt_03` i tylko tam mieszka praca ręczna.
6. Po zweryfikowaniu wszystkich odcinków uruchomić `tbbt_03_annotations.ipynb`.